In [1]:
!pip install -qU  langchain-huggingface text-generation transformers google-search-results numexpr langchainhub sentencepiece jinja2 bitsandbytes accelerate

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.0 MB/s eta 0:00:00


In [2]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFacePipeline
from transformers import AutoTokenizer, pipeline
import transformers
import accelerate
from langchain_core.prompts import PromptTemplate
from IPython.display import display, Markdown, HTML
from langchain_core.output_parsers import StrOutputParser

In [ ]:
import getpass
import os

os.environ["HF_TOKEN"] = getpass.getpass(
    ""
)

In [4]:
model = "Qwen/Qwen2.5-3B-Instruct"
tokenizer=AutoTokenizer.from_pretrained(model,trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
# Create the transformers pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1000,
    device_map = "auto",
    return_full_text=False, # THIS IS THE KEY FIX,
    clean_up_tokenization_spaces=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [13]:
llm = HuggingFacePipeline(pipeline=pipe, model_kwargs={"temperature": 7})

In [14]:
chat_model = ChatHuggingFace(llm=llm)


In [26]:
prompt = "I want to watch a sci-fi film. Recommend me a sci-fi film with deep meaning and arthouse style."
response = chat_model.invoke(prompt)
display(Markdown(response.content))

Certainly! If you're looking for a sci-fi film with deep meaning and an arthouse style, I'd recommend "Blade Runner 2049" directed by Denis Villeneuve. This film is known for its rich narrative, philosophical themes, and visually stunning cinematography.

"Blade Runner 2049" follows the story of K, a blade runner tasked with hunting down and retiring replicants who have escaped Earth's atmosphere. The film delves into complex questions about humanity, identity, and the nature of consciousness, making it a profound and thought-provoking experience.

If you prefer something a bit more recent and perhaps even more abstract, you might enjoy "Arrival," a 2016 film directed by Denis Villeneuve. It tells the story of a linguist recruited to communicate with extraterrestrial beings who have landed on Earth, focusing on the psychological impact of these encounters and the complexities of language and time.

Both films offer unique insights into the human condition and the future of technology, making them excellent choices for someone seeking a deeper exploration within the genre of science fiction.

In [16]:
prompt_template=PromptTemplate(input_variables=["cuisine"],
                               template="I want to open a restaurant for {cuisine} food. Suggest a fency name for this")

In [17]:
prompt_=prompt_template.format(cuisine="middle eastern")

In [18]:
prompt_

'I want to open a restaurant for middle eastern food. Suggest a fency name for this'

In [19]:
response = chat_model.invoke(prompt_)
response.content

"Certainly! Here are a few fancy names that could work well for your Middle Eastern-inspired restaurant:\n\n1. **Noura's Niche**\n2. **Al-Ma'at Restaurant**\n3. **Le Za'atar Bistro**\n4. **The Spice Pavilion**\n5. **Arabica's Altar**\n6. **Leila's Larder**\n7. **Abu's Alchemy Kitchen**\n8. **Bougainvillea Breads**\n9. **The Golden Crescent Café**\n10. **Saffron's Sanctuary**\n\nFeel free to choose one that resonates with you or customize it further based on your vision and branding preferences!"

## Chain way

In [21]:
prompt_template= """I want to open a restaurant for {cuisine} food. Suggest a fency name for this"""

In [22]:
prompt_ = PromptTemplate(input_variables=["cuisine"],
                               template=prompt_template)

In [23]:
chain = prompt_ | chat_model | StrOutputParser()

In [24]:
out = chain.invoke({"cuisine":"middle eastern"})

In [25]:
display(HTML("<h3 style='color: #4285F4;'>🚀 Here is the answer</h3>"))
display(Markdown(f"## Final Result\n{out}"))

## Final Result
Certainly! Here are a few creative and enticing names for your Middle Eastern restaurant:

1. **Lebanese Breeze**
2. **Arabian Spice**
3. **Mediterranean Delights**
4. **Golden Crescent**
5. **Halal Haven**
6. **Saffron Shadows**
7. **Taj Mahal Cuisine**
8. **Feta Fortune**
9. **Cedar Grove**
10. **Rice Revival**

Choose the one that best fits your brand identity and resonates with your target audience. Enjoy opening your restaurant!